# Shared data module
This notebook is the short version of `data.py`, so we all preprocess MNIST the same way. Run it from the assignment folder (`python -m jupyter nbconvert --execute` is not needed, just open it). It needs `torch`, `torchvision` and `matplotlib`; the MNIST files are downloaded once into `build/data`.

In [1]:
import data as D
import torch

D.summary()

train gray (60000, 784) 0.0 1.0
train bin  (60000, 784) 0 1 mean pixel 0.1323 labels (60000,)
test gray (10000, 784) 0.0 1.0
test bin  (10000, 784) 0 1 mean pixel 0.1342 labels (10000,)


The arrays are float32 of shape (N, 784) and the pixel index is row-major (`p = row * 28 + col`). Task 1 and task 2 use `load_binarized`, task 3 uses `load_gray` because the GAN should see the original grayscale images.

In [2]:
xb, yb = D.load_binarized("train")
xg, _ = D.load_gray("train")
xt, yt = D.load_binarized("test")
print(xb.shape, xt.shape, xg.shape)
print("unique values in xb:", torch.unique(torch.from_numpy(xb)).tolist())

(60000, 784) (10000, 784) (60000, 784)


unique values in xb: [0.0, 1.0]


A quick look at twelve binarised training images. Binarisation is a fixed threshold at 0.5, so the same digit always gives the same 0/1 image.

In [3]:
D.show_grid(xb[:12], 2, 6, path="build/demo_grid.png", title="binarised training images")

'build/demo_grid.png'

The threshold is a design choice, not a law. Raising it removes ink (and part of the stroke), lowering it keeps shadows and background noise. The numbers below are the fraction of pixels that are 1:

In [4]:
for thr in (0.3, 0.5, 0.7):
    x, _ = D.load_binarized("train", threshold=thr)
    print("threshold", thr, "mean pixel", round(float(x.mean()), 4))

threshold 0.3 mean pixel 0.1511


threshold 0.5 mean pixel 0.1323


threshold 0.7 mean pixel 0.1116


One training batch, the way the autoregressive models consume it:

In [5]:
batch = torch.from_numpy(xb[:128])
print(batch.shape, batch.dtype)
print("pixels set in this batch:", int(batch.sum().item()))

torch.Size([128, 784]) torch.float32
pixels set in this batch: 12897


The order of the pixels matters for the autoregressive models, so `data.py` also has `pixel_order("row")`, `pixel_order("col")`, `apply_order` and `undo_order`. For the shared comparison table each task should log: preprocessing, learning rate, batch size, epochs, optimiser, seed, train time, sampling time for 100 images, and test bits/dim. The FVSBN numbers are in `build/results_summary.csv`.